Changing my current work directory to project root , so i can get the parquet files without actually changing the code in transformation .

In [83]:
import os
from pathlib import Path

# Find project root directory (directory containing raw/ or config.py)
PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / "raw").exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent

# Change current working directory to project root
os.chdir(PROJECT_ROOT)

print(f"Working directory set to project root: {os.getcwd()}")

Working directory set to project root: c:\Users\Likhita\Desktop\pokemon-etl


# Pokémon ETL — Transformation Demo

This notebook shows how the raw Parquet data is cleaned, transformed and prepared for database loading.

## Transformation Flow

Raw Parquet → Transform → Clean → Derived Attributes → Validation

## 1. Load Raw Data

First, we load the raw Pokémon, species and move data from Parquet files..

In [84]:
import sys
from pathlib import Path

import pandas as pd


# get project root
PROJECT_ROOT = Path.cwd()

if not (PROJECT_ROOT / "transform.py").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.append(str(PROJECT_ROOT))

from transform_runner import load_parquet
from transform import (
    transform_data,
    clean_data,
    add_derived_attributes,
    transform_types,
    transform_abilities,
    transform_species,
    add_special_status,
    transform_moves,
    transform_pokemon_moves,
    get_id_from_url
)

In [85]:
# loading raw parquet files
pokemon_data = load_parquet("pokemon")
species_data = load_parquet("species")
move_data = load_parquet("moves")

print("Pokemon:", len(pokemon_data))
print("Species:", len(species_data))
print("Moves:", len(move_data))

Pokemon: 1351
Species: 1025
Moves: 833


## POKEMON DATA BEFORE TRANSFORMATION

The raw data we stored in the parquet file

In [86]:
pokemon_data[:5]

[{'id': 1,
  'name': 'bulbasaur',
  'base_experience': 64,
  'height': 7,
  'is_default': True,
  'order': 1,
  'weight': 69,
  'abilities': [{'is_hidden': False,
    'slot': 1,
    'ability': {'name': 'overgrow',
     'url': 'https://pokeapi.co/api/v2/ability/65/'}},
   {'is_hidden': True,
    'slot': 3,
    'ability': {'name': 'chlorophyll',
     'url': 'https://pokeapi.co/api/v2/ability/34/'}}],
  'past_abilities': [{'generation': {'name': 'generation-iv',
     'url': 'https://pokeapi.co/api/v2/generation/4/'},
    'abilities': [{'is_hidden': True, 'slot': 3, 'ability': None}]}],
  'forms': [{'name': 'bulbasaur',
    'url': 'https://pokeapi.co/api/v2/pokemon-form/1/'}],
  'game_indices': [{'game_index': 153,
    'version': {'name': 'red', 'url': 'https://pokeapi.co/api/v2/version/1/'}},
   {'game_index': 153,
    'version': {'name': 'blue',
     'url': 'https://pokeapi.co/api/v2/version/2/'}},
   {'game_index': 153,
    'version': {'name': 'yellow',
     'url': 'https://pokeapi.co/a

## 2. Transform Pokémon Data

The raw Pokémon response contains nested data, so we extract only the fields required for the database.

In [87]:
pokemon_df = transform_data(pokemon_data)

pokemon_df.head()

,pokemon_id,species_id,pokemon_name,form_name,is_default_form,height_m,weight_kg,base_experience,hp,attack,defense,special_attack,special_defense,speed
0,1,1,bulbasaur,bulbasaur,True,0.7,6.9,64.0,45,49,49,65,65,45
1,2,2,ivysaur,ivysaur,True,1.0,13.0,142.0,60,62,63,80,80,60
2,3,3,venusaur,venusaur,True,2.0,100.0,236.0,80,82,83,100,100,80
3,4,4,charmander,charmander,True,0.6,8.5,62.0,39,52,43,60,50,65
4,5,5,charmeleon,charmeleon,True,1.1,19.0,142.0,58,64,58,80,65,80


## How I structured the Pokémon Data

The API response contains nested data, so I extract the required fields and convert the stats into separate columns.

In [88]:
from config import STAT_COLUMNS 

In [89]:
from config import STAT_COLUMNS

records = []

for pokemon in pokemon_data[:5]:

    species_info = pokemon.get("species") or {}

    record = {
        "pokemon_id": pokemon.get("id"),
        "species_id": get_id_from_url(species_info),
        "pokemon_name": pokemon.get("name"),
        "form_name": (
            (pokemon.get("forms") or [{}])[0].get("name")
        ),
        "is_default_form": pokemon.get("is_default", True),
        "height_m": pokemon.get("height", 0) / 10,
        "weight_kg": pokemon.get("weight", 0) / 10,
        "base_experience": pokemon.get("base_experience")
    }

    # Create stat columns
    for stat in STAT_COLUMNS:
        record[stat] = None

    # Extract stat values
    for item in pokemon.get("stats", []):
        stat_name = item["stat"]["name"].replace("-", "_")

        if stat_name in STAT_COLUMNS:
            record[stat_name] = item["base_stat"]

    records.append(record)

structured_demo_df = pd.DataFrame(records)

structured_demo_df

,pokemon_id,species_id,pokemon_name,form_name,is_default_form,height_m,weight_kg,base_experience,hp,attack,defense,special_attack,special_defense,speed
0,1,1,bulbasaur,bulbasaur,True,0.7,6.9,64,45,49,49,65,65,45
1,2,2,ivysaur,ivysaur,True,1.0,13.0,142,60,62,63,80,80,60
2,3,3,venusaur,venusaur,True,2.0,100.0,236,80,82,83,100,100,80
3,4,4,charmander,charmander,True,0.6,8.5,62,39,52,43,60,50,65
4,5,5,charmeleon,charmeleon,True,1.1,19.0,142,58,64,58,80,65,80


In [90]:
pokemon_df=pd.DataFrame(records)

pokemon_df.head()

,pokemon_id,species_id,pokemon_name,form_name,is_default_form,height_m,weight_kg,base_experience,hp,attack,defense,special_attack,special_defense,speed
0,1,1,bulbasaur,bulbasaur,True,0.7,6.9,64,45,49,49,65,65,45
1,2,2,ivysaur,ivysaur,True,1.0,13.0,142,60,62,63,80,80,60
2,3,3,venusaur,venusaur,True,2.0,100.0,236,80,82,83,100,100,80
3,4,4,charmander,charmander,True,0.6,8.5,62,39,52,43,60,50,65
4,5,5,charmeleon,charmeleon,True,1.1,19.0,142,58,64,58,80,65,80


## Creating a dummy inconsistent data 

In [91]:
# Create a copy for cleaning demonstration
dirty_df = pokemon_df.copy()

# Cast height_m to object so it can accept string values with whitespace
dirty_df["height_m"] = dirty_df["height_m"].astype(object)

# Introduce sample inconsistencies
dirty_df.loc[0, "height_m"] = " 0.7 "      # Leading/trailing whitespace
dirty_df.loc[1, "pokemon_name"] = None     # Missing value
dirty_df.loc[2, "weight_kg"] = -10         # Invalid negative value
dirty_df.loc[3, "height_m"] = "invalid"    # Non-numeric string

# Add a duplicate row
dirty_df = pd.concat(
    [dirty_df, dirty_df.iloc[[0]]],
    ignore_index=True
)

dirty_df.head(6)

,pokemon_id,species_id,pokemon_name,form_name,is_default_form,height_m,weight_kg,base_experience,hp,attack,defense,special_attack,special_defense,speed
0,1,1,bulbasaur,bulbasaur,True,0.7,6.9,64,45,49,49,65,65,45
1,2,2,NaN,ivysaur,True,1.0,13.0,142,60,62,63,80,80,60
2,3,3,venusaur,venusaur,True,2.0,-10.0,236,80,82,83,100,100,80
3,4,4,charmander,charmander,True,invalid,8.5,62,39,52,43,60,50,65
4,5,5,charmeleon,charmeleon,True,1.1,19.0,142,58,64,58,80,65,80
5,1,1,bulbasaur,bulbasaur,True,0.7,6.9,64,45,49,49,65,65,45


## 3. Clean the Data

I remove duplicates, convert the required columns to numeric values, handle missing values and apply basic validation rules.

In [92]:
pokemon_df = clean_data(pokemon_df)

pokemon_df.head()

Duplicates removed: 0


,pokemon_id,species_id,pokemon_name,form_name,is_default_form,height_m,weight_kg,base_experience,hp,attack,defense,special_attack,special_defense,speed
0,1,1,bulbasaur,bulbasaur,True,0.7,6.9,64,45,49,49,65,65,45
1,2,2,ivysaur,ivysaur,True,1.0,13.0,142,60,62,63,80,80,60
2,3,3,venusaur,venusaur,True,2.0,100.0,236,80,82,83,100,100,80
3,4,4,charmander,charmander,True,0.6,8.5,62,39,52,43,60,50,65
4,5,5,charmeleon,charmeleon,True,1.1,19.0,142,58,64,58,80,65,80


## How I cleaned the data

In [93]:
# remove duplicate pokemon
cleaned_df=dirty_df.copy()

cleaned_df= cleaned_df.drop_duplicates(
    subset=["pokemon_id"]
)


# an duplicate record with pokemon id 1 is removed
cleaned_df.head()

,pokemon_id,species_id,pokemon_name,form_name,is_default_form,height_m,weight_kg,base_experience,hp,attack,defense,special_attack,special_defense,speed
0,1,1,bulbasaur,bulbasaur,True,0.7,6.9,64,45,49,49,65,65,45
1,2,2,NaN,ivysaur,True,1.0,13.0,142,60,62,63,80,80,60
2,3,3,venusaur,venusaur,True,2.0,-10.0,236,80,82,83,100,100,80
3,4,4,charmander,charmander,True,invalid,8.5,62,39,52,43,60,50,65
4,5,5,charmeleon,charmeleon,True,1.1,19.0,142,58,64,58,80,65,80


In [94]:
from config import  NUMERIC_COLUMNS , CATEGORICAL_COLUMNS , DEFAULT_CATEGORICAL_FILL

In [95]:
# convert numeric columns
for column in NUMERIC_COLUMNS:
    cleaned_df[column] = pd.to_numeric(
        cleaned_df[column],
        errors="coerce"
    ).fillna(0)

    cleaned_df[column]=cleaned_df[column].clip(lower=0)

# changing the datatype of numeric columns to numeric and  if there is any non numeric value , we get invalid 
# filling invalid with 0 
# and also if any negative value , change the value to 0
cleaned_df.head()

,pokemon_id,species_id,pokemon_name,form_name,is_default_form,height_m,weight_kg,base_experience,hp,attack,defense,special_attack,special_defense,speed
0,1,1,bulbasaur,bulbasaur,True,0.7,6.9,64,45,49,49,65,65,45
1,2,2,NaN,ivysaur,True,1.0,13.0,142,60,62,63,80,80,60
2,3,3,venusaur,venusaur,True,2.0,0.0,236,80,82,83,100,100,80
3,4,4,charmander,charmander,True,0.0,8.5,62,39,52,43,60,50,65
4,5,5,charmeleon,charmeleon,True,1.1,19.0,142,58,64,58,80,65,80


In [96]:
# fill missing categorical values
cleaned_df[CATEGORICAL_COLUMNS] = (
    cleaned_df[CATEGORICAL_COLUMNS]
    .fillna(DEFAULT_CATEGORICAL_FILL)
)

#if any null value in categorical columns then we will fill with unknown
cleaned_df.head()

,pokemon_id,species_id,pokemon_name,form_name,is_default_form,height_m,weight_kg,base_experience,hp,attack,defense,special_attack,special_defense,speed
0,1,1,bulbasaur,bulbasaur,True,0.7,6.9,64,45,49,49,65,65,45
1,2,2,Unknown,ivysaur,True,1.0,13.0,142,60,62,63,80,80,60
2,3,3,venusaur,venusaur,True,2.0,0.0,236,80,82,83,100,100,80
3,4,4,charmander,charmander,True,0.0,8.5,62,39,52,43,60,50,65
4,5,5,charmeleon,charmeleon,True,1.1,19.0,142,58,64,58,80,65,80


## DIRTY DATA VS CLEANED DATA

In [97]:
dirty_df.head()


,pokemon_id,species_id,pokemon_name,form_name,is_default_form,height_m,weight_kg,base_experience,hp,attack,defense,special_attack,special_defense,speed
0,1,1,bulbasaur,bulbasaur,True,0.7,6.9,64,45,49,49,65,65,45
1,2,2,NaN,ivysaur,True,1.0,13.0,142,60,62,63,80,80,60
2,3,3,venusaur,venusaur,True,2.0,-10.0,236,80,82,83,100,100,80
3,4,4,charmander,charmander,True,invalid,8.5,62,39,52,43,60,50,65
4,5,5,charmeleon,charmeleon,True,1.1,19.0,142,58,64,58,80,65,80


In [98]:

cleaned_df.head()

,pokemon_id,species_id,pokemon_name,form_name,is_default_form,height_m,weight_kg,base_experience,hp,attack,defense,special_attack,special_defense,speed
0,1,1,bulbasaur,bulbasaur,True,0.7,6.9,64,45,49,49,65,65,45
1,2,2,Unknown,ivysaur,True,1.0,13.0,142,60,62,63,80,80,60
2,3,3,venusaur,venusaur,True,2.0,0.0,236,80,82,83,100,100,80
3,4,4,charmander,charmander,True,0.0,8.5,62,39,52,43,60,50,65
4,5,5,charmeleon,charmeleon,True,1.1,19.0,142,58,64,58,80,65,80


## 4. Add Derived Attributes

These columns are calculated from the Pokémon data instead of coming directly from the API.

In [99]:
pokemon_df = add_derived_attributes(pokemon_df)

pokemon_df[
    [
        "pokemon_name",
        "total_base_stats",
        "offensive_power",
        "defensive_power",
        "speed_percentile",
        "battle_style",
        "stat_specialization",
        "size_class"
    ]
].head(5)

,pokemon_name,total_base_stats,offensive_power,defensive_power,speed_percentile,battle_style,stat_specialization,size_class
0,bulbasaur,318,114,114,20.0,Balanced,special_attack,Small
1,ivysaur,405,142,143,40.0,Balanced,special_attack,Medium
2,venusaur,525,182,183,90.0,Balanced,special_attack,Large
3,charmander,309,112,93,60.0,Balanced,speed,Small
4,charmeleon,405,144,123,90.0,Offensive,special_attack,Large


## 5. Transform Types and Abilities

Types and abilities are multi-valued fields, so they are converted into separate relationship tables.

In [100]:
types_df = transform_types(pokemon_data)
abilities_df = transform_abilities(pokemon_data)

print("Types")
display(types_df.head())

print("Abilities")
display(abilities_df.head())

Types


,pokemon_id,type_name,slot
0,1,grass,1
1,1,poison,2
2,2,grass,1
3,2,poison,2
4,3,grass,1


Abilities


,pokemon_id,ability_name,slot,is_hidden
0,1,overgrow,1,False
1,1,chlorophyll,3,True
2,2,overgrow,1,False
3,2,chlorophyll,3,True
4,3,overgrow,1,False


## 6. Transform Species Data

Species data contains information such as generation, habitat, growth rate and special status fields.

In [101]:
species_df = transform_species(species_data)

species_df = add_special_status(species_df)

species_df[
    [
        "species_id",
        "pokemon_category",
        "generation",
        "color",
        "shape",
        "habitat",
        "capture_rate",
        "base_happiness",
        "growth_rate",
        "gender_rate",
        "hatch_counter",
        "egg_group_1",
        "egg_group_2",
        "is_baby",
        "is_legendary",
        "is_mythical",
        "special_status"
    ]
].head()

,species_id,pokemon_category,generation,color,shape,habitat,capture_rate,base_happiness,growth_rate,gender_rate,hatch_counter,egg_group_1,egg_group_2,is_baby,is_legendary,is_mythical,special_status
0,2,Seed Pokémon,generation-i,green,quadruped,grassland,45,70,medium-slow,1,20,monster,plant,False,False,False,Normal
1,842,Apple Nectar Pokémon,generation-viii,green,quadruped,NaN,45,50,slow-then-very-fast,4,20,plant,dragon,False,False,False,Normal
2,381,Eon Pokémon,generation-iii,blue,wings,waters-edge,3,90,slow,0,120,no-eggs,NaN,False,True,False,Legendary
3,674,Playful Pokémon,generation-vi,white,upright,NaN,220,70,medium,4,25,ground,humanshape,False,False,False,Normal
4,883,Fossil Pokémon,generation-viii,blue,fish,NaN,45,50,slow,-1,35,no-eggs,NaN,False,False,False,Normal


## 7. Transform Moves

Move information is kept separately because moves have their own attributes such as power, accuracy, PP and damage class.

In [102]:
moves_df = transform_moves(move_data)

moves_df.head()

,move_id,move_name,move_type,power,accuracy,pp,priority,damage_class,generation
0,1,pound,normal,40.0,100.0,35,0,physical,generation-i
1,10,scratch,normal,40.0,100.0,35,0,physical,generation-i
2,100,teleport,psychic,NaN,NaN,20,-6,status,generation-i
3,101,night-shade,ghost,NaN,100.0,15,0,special,generation-i
4,102,mimic,normal,NaN,NaN,10,0,status,generation-i


## 8. Pokémon-Move Relationship

A Pokémon can learn many moves and a move can be learned by many Pokémon, so we create a separate relationship table.

In [103]:
pokemon_moves_df = transform_pokemon_moves(pokemon_data)

pokemon_moves_df.head()

,pokemon_id,move_id,move_name,learn_method,level_learned_at,version_group
0,1,13,razor-wind,egg,0,gold-silver
1,1,13,razor-wind,egg,0,crystal
2,1,14,swords-dance,machine,0,red-blue
3,1,14,swords-dance,machine,0,yellow
4,1,14,swords-dance,tutor,0,emerald


## 9. Transformation Summary

All required DataFrames are now ready for validation and database loading.

In [104]:
datasets = {
    "pokemon": pokemon_df,
    "types": types_df,
    "abilities": abilities_df,
    "species": species_df,
    "moves": moves_df,
    "pokemon_moves": pokemon_moves_df
}

for name, df in datasets.items():
    print(
        f"{name}: {len(df)} rows, "
        f"{len(df.columns)} columns"
    )

pokemon: 5 rows, 21 columns
types: 2116 rows, 3 columns
abilities: 2943 rows, 4 columns
species: 1025 rows, 17 columns
moves: 833 rows, 9 columns
pokemon_moves: 638321 rows, 6 columns


## Transformation Complete

The raw data is now cleaned, structured and enriched for the validation and loading stages.